In [1]:
import glob
import math
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
# Download list of Olink genes

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/preprocessed/proteomics_genes.txt

olink_genes = pl.read_csv('proteomics_genes.txt', has_header=False).rename({'column_1': 'region'})
olink_genes

Error: path
"/home/dnanexus/ukbgym/utils/average_pheno_per_variant/proteomics_genes.txt"
already exists but -f/--overwrite was not set


region
str
"""ENSG00000266967"""
"""ENSG00000114779"""
"""ENSG00000097007"""
"""ENSG00000060971"""
"""ENSG00000157766"""
…
"""ENSG00000173465"""
"""ENSG00000105428"""
"""ENSG00000188372"""


In [3]:
# Download annotation file for variant-gene mapping
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym_with_mane.parquet -o /home/dnanexus/data_dir/

anno = pl.read_parquet(
    '/home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquet', 
    columns=['id', 'region']
)

anno

Error: path
"/home/dnanexus/data_dir/annotations_fillna_ukbgym_with_mane.parquet" already
exists but -f/--overwrite was not set


id,region
str,str
"""chr8:134702216:A:G""","""ENSG00000066827"""
"""chr19:47846298:C:T""","""ENSG00000105392"""
"""chr17:49763088:G:A""","""ENSG00000121104"""
"""chr8:38172782:G:C""","""ENSG00000175324"""
"""chr1:174771515:C:T""","""ENSG00000152061"""
…,…
"""chr11:70472380:A:T""","""ENSG00000162105"""
"""chr8:16174283:C:T""","""ENSG00000038945"""
"""chr2:215381006:G:C""","""ENSG00000115414"""


In [4]:
anno['region'].value_counts(sort=True).join(olink_genes, on='region', how='semi')

region,count
str,u64
"""ENSG00000174469""",605106
"""ENSG00000185008""",444475
"""ENSG00000189283""",435466
"""ENSG00000021645""",385076
"""ENSG00000149972""",376497
…,…
"""ENSG00000179889""",1616
"""ENSG00000160221""",669
"""ENSG00000267368""",611


In [5]:
# prot_file = "cauc_cov_regression_90pcs_prs"  # covariates and PRS corrected
# prot_file = "cauc_protrider_lite_prs_rint"   # PROTRIDER corrected (RINT)
prot_file = "cauc_protrider_lite_prs_t_df"     # PROTRIDER corrected (T distribution)

# Download Olink:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/olink/adjusted/{prot_file}.parquet -o /home/dnanexus/data_dir/

phenos = pl.read_parquet(f'/home/dnanexus/data_dir/{prot_file}.parquet')

olink_genes_w_data = list(set(phenos.columns).intersection(set(olink_genes['region'])).intersection(set(anno['region'])))

phenos = phenos.select(['sample'] + olink_genes_w_data)

long_phenos = (
    phenos
    .unpivot(
        index='sample',
        on=olink_genes_w_data,
        variable_name='phenotype',
        value_name='pheno_value'
    )

    .with_columns(
        region = pl.col('phenotype'),
        phenotype = pl.col('phenotype') + '_olink',
    )
    .drop_nulls()
)

print(long_phenos['phenotype'].value_counts(sort=True))
long_phenos

Error: path "/home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df.parquet"
already exists but -f/--overwrite was not set
shape: (2_661, 2)
┌───────────────────────┬───────┐
│ phenotype             ┆ count │
│ ---                   ┆ ---   │
│ str                   ┆ u64   │
╞═══════════════════════╪═══════╡
│ ENSG00000197461_olink ┆ 39208 │
│ ENSG00000108622_olink ┆ 39208 │
│ ENSG00000073849_olink ┆ 39208 │
│ ENSG00000172016_olink ┆ 39208 │
│ ENSG00000146648_olink ┆ 39208 │
│ …                     ┆ …     │
│ ENSG00000102837_olink ┆ 31597 │
│ ENSG00000131050_olink ┆ 31502 │
│ ENSG00000111405_olink ┆ 30953 │
│ ENSG00000163131_olink ┆ 30432 │
│ ENSG00000170373_olink ┆ 29106 │
└───────────────────────┴───────┘


sample,phenotype,pheno_value,region
str,str,f64,str
"""5645319""","""ENSG00000099795_olink""",-1.450133,"""ENSG00000099795"""
"""5959139""","""ENSG00000099795_olink""",-1.474268,"""ENSG00000099795"""
"""5732867""","""ENSG00000099795_olink""",-1.425497,"""ENSG00000099795"""
"""2074480""","""ENSG00000099795_olink""",0.375579,"""ENSG00000099795"""
"""4659532""","""ENSG00000099795_olink""",1.032297,"""ENSG00000099795"""
…,…,…,…
"""3191148""","""ENSG00000197959_olink""",0.489134,"""ENSG00000197959"""
"""1807196""","""ENSG00000197959_olink""",1.998566,"""ENSG00000197959"""
"""4223555""","""ENSG00000197959_olink""",2.289093,"""ENSG00000197959"""


In [6]:
# Download genotype (long gt) file
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/gt_long.parquet -o /home/dnanexus/data_dir/

long_gt = (
    pl.scan_parquet('/home/dnanexus/data_dir/gt_long.parquet')
    .select(['id', 'sample', 'gt'])
    .filter(pl.col('gt')==1)
)
long_gt.head().collect()

Error: path "/home/dnanexus/data_dir/gt_long.parquet" already exists but
-f/--overwrite was not set


id,sample,gt
str,str,i8
"""chr10:20020006:C:T""","""5317620""",1
"""chr10:20020007:TTTTCTTGC:T""","""5546988""",1
"""chr10:20020010:T:C""","""2793793""",1
"""chr10:20020014:G:A""","""1722739""",1
"""chr10:20020014:G:A""","""1139673""",1


In [ ]:
output_dir = '/home/dnanexus/data_dir/appv_phenos/'
!mkdir -p {output_dir}

CHUNK_SIZE = 100
total_genes = len(olink_genes_w_data)
num_chunks = math.ceil(total_g  enes / CHUNK_SIZE)

print(f"Processing {total_genes} genes in {num_chunks} chunks...")

# Process in Batches
for i in tqdm(range(0, total_genes, CHUNK_SIZE)):
    # 1. Define the current batch of genes
    chunk_genes = olink_genes_w_data[i : i + CHUNK_SIZE]
    chunk_phenos = [f"{g}_olink" for g in chunk_genes]

# for olink_gene in tqdm(olink_genes_w_data):
    print(f"Processing chunk starting at index: {i}")
    
    (
        anno.filter(pl.col('region').is_in(chunk_genes))
        .lazy()
        .join(
            long_gt,
            on='id',
            how='inner'
        )

        .join(
            long_phenos.filter(pl.col('phenotype').is_in(chunk_phenos)).lazy(),
            on=['sample', 'region'],
            how='inner'
        )

        .group_by(['id', 'region', 'phenotype'])
        .agg(
            n_individuals = pl.len().cast(pl.Int32),
            mean_pheno_value = pl.col('pheno_value').mean().cast(pl.Float32),
            std_pheno_value = pl.col('pheno_value').std().cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(method="max")
                .cast(pl.Float32),
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len()
            ).cast(pl.Float32),
        )

        .sink_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk{i}.parquet')
        # .collect(engine='streaming')
    )

Processing 2661 genes in 27 chunks...


  0%|          | 0/27 [00:00<?, ?it/s]

Processing chunk starting at index: 0


  4%|▎         | 1/27 [00:14<06:06, 14.11s/it]

Processing chunk starting at index: 100


...

 93%|█████████▎| 25/27 [05:38<00:26, 13.48s/it]

Processing chunk starting at index: 2500


 96%|█████████▋| 26/27 [05:52<00:13, 13.51s/it]

Processing chunk starting at index: 2600


100%|██████████| 27/27 [06:05<00:00, 13.55s/it]


## Consolidate parquet

In [8]:
pl.read_parquet(f'{output_dir}/olink_all_genes_EURunrelated_appv_chunk0.parquet')

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:62215258:A:G""","""ENSG00000182010""","""ENSG00000182010_olink""",2,-0.1991,0.458364,294178.0,0.385238
"""chr10:62203465:C:T""","""ENSG00000182010""","""ENSG00000182010_olink""",1,0.711822,null,624719.0,0.818094
"""chr10:62212343:T:C""","""ENSG00000182010""","""ENSG00000182010_olink""",1,1.711007,null,743123.0,0.973149
"""chr14:102951783:G:C""","""ENSG00000198752""","""ENSG00000198752_olink""",3,-0.021044,1.025009,370245.0,0.484851
"""chr14:102951329:G:T""","""ENSG00000198752""","""ENSG00000198752_olink""",1,-0.513669,null,187301.0,0.245278
…,…,…,…,…,…,…,…
"""chr3:57761940:G:A""","""ENSG00000163681""","""ENSG00000163681_olink""",1,-1.243244,null,52936.0,0.069322
"""chr3:57776358:C:T""","""ENSG00000163681""","""ENSG00000163681_olink""",1,0.038934,null,396987.0,0.51987
"""chr3:57768522:C:G""","""ENSG00000163681""","""ENSG00000163681_olink""",1,-0.215752,null,287572.0,0.376587


In [9]:
combined_output_file = f"/home/dnanexus/data_dir/{prot_file}_all_genes_EURunrelated_appv_percentiles.parquet"

# 1. Get list of files manually
files = glob.glob(output_dir)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(combined_output_file, engine='streaming')
print("Done.")

Found 1 files.
Streaming to disk...
Done.


In [10]:
!dx upload {combined_output_file} --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/ukbgym/avg_pheno_per_var/

[===========================================================>] Uploaded 313,246,696 of 313,246,696 bytes (100%) /home/dnanexus/data_dir/cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentiles.parquet
ID                                file-J609GQ0Jg0y54jg3BxPxQ0Pg
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/ukbgym/avg_pheno_per_var
Name                              cauc_protrider_lite_prs_t_df_all_genes_EURunrelated_appv_percentil
                                  es.parquet
State                             closing
Visibility                        visible
Types                             -
Properties                        -
Tags                              -
Outgoing links                    -
Created                           Mon Feb  2 12:19:12 2026
Created by                        shubhankar
 via the job                      job-J604f08Jg0yPk2gGpP0q1z3y
L

In [11]:
a = pl.scan_parquet(combined_output_file)
a.head().collect()

id,region,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,str,i32,f32,f32,f32,f32
"""chr10:62215258:A:G""","""ENSG00000182010""","""ENSG00000182010_olink""",2,-0.1991,0.458364,294178.0,0.385238
"""chr10:62203465:C:T""","""ENSG00000182010""","""ENSG00000182010_olink""",1,0.711822,null,624719.0,0.818094
"""chr10:62212343:T:C""","""ENSG00000182010""","""ENSG00000182010_olink""",1,1.711007,null,743123.0,0.973149
"""chr14:102951783:G:C""","""ENSG00000198752""","""ENSG00000198752_olink""",3,-0.021044,1.025009,370245.0,0.484851
"""chr14:102951329:G:T""","""ENSG00000198752""","""ENSG00000198752_olink""",1,-0.513669,null,187301.0,0.245278


In [12]:
a.select(['region']).collect()['region'].unique()

region
str
"""ENSG00000103343"""
"""ENSG00000157823"""
"""ENSG00000171824"""
"""ENSG00000163736"""
"""ENSG00000132740"""
…
"""ENSG00000182512"""
"""ENSG00000138760"""
"""ENSG00000105825"""
